# Part 2 - Is v0 more accurate than Savant?

In Part 1 I showed that v0 faithfully reproduces Savant's xwOBA. But reproducing a metric isn't the same as improving on it. So here's the honest test of any xwOBA number: does *this* year's value predict *next* year's actual wOBA?

I line up three predictors against next-season wOBA — my v0 model, public Savant xwOBA, and a naive "just use last year's actual wOBA" baseline — on the 1,058 player-pairs with 100+ PA in both years. Everything below comes out of `results/benchmark/`.

In [ ]:
# --- setup: find repo root, load helpers (run this first) ---
import sys, json
from pathlib import Path
import polars as pl
from IPython.display import Image, Markdown, display

pl.Config.set_tbl_rows(30)
here = Path.cwd()
ROOT = next((p for p in [here, *here.parents] if (p / "config.yaml").exists()), here)
sys.path.insert(0, str(ROOT))
RESULTS = ROOT / "results"

def jload(rel):
    return json.loads((RESULTS / rel).read_text())

def show_fig(rel, width=680):
    p = RESULTS / rel
    return Image(filename=str(p), width=width) if p.exists() else Markdown(f"_missing figure: {rel}_")

print("repo root:", ROOT)
print("results:  ", RESULTS, "(exists)" if RESULTS.exists() else "(MISSING)")

## Pooled predictive accuracy

First pass: how well does each predictor correlate (Pearson r) with next season's actual wOBA?

In [ ]:
bm = jload("benchmark/benchmark_metrics.json")
p = bm["pooled"]
pl.DataFrame([
    {"predictor": "v0 model", "r": round(p["model"]["r"], 4), "rmse_calibrated": round(p["model"]["rmse_calibrated"], 4)},
    {"predictor": "Savant",   "r": round(p["savant"]["r"], 4), "rmse_calibrated": round(p["savant"]["rmse_calibrated"], 4)},
    {"predictor": "naive (last-yr wOBA)", "r": round(p["naive"]["r"], 4), "rmse_calibrated": round(p["naive"]["rmse_calibrated"], 4)},
])

## Is the v0 − Savant gap actually real?

v0 and Savant look close, but "close" needs a number. Let me bootstrap the difference in their correlations and see whether it's distinguishable from zero.

In [ ]:
b = bm["model_vs_savant_bootstrap"]
print("n_pairs:", bm["n_pairs"])
print(f"r_model - r_savant = {b['r_model_minus_r_savant']:+.4f}")
print(f"95% CI: [{b['ci95'][0]:+.4f}, {b['ci95'][1]:+.4f}]")
print("fraction of bootstraps where model beats Savant:", b["frac_model_better"])
print("verdict:", bm["verdict"])
Markdown("**The CI straddles 0 → statistical parity. v0 does not beat Savant, but both clearly beat the naive baseline, so v0 is genuine xwOBA.**")

## The predictive-accuracy picture

Same story, in one plot:

In [ ]:
show_fig("benchmark/figures/predictive_accuracy.png")

## So why parity, and what does it mean?

The way I read this: the 3-feature model is sitting right at Savant's information ceiling. Fiddling with the model won't beat Savant — the only way past this wall is to feed it inputs Savant doesn't have (spray direction, batter handedness), which is exactly what a future **v1** would add.

That gives me a concrete target for v1: pooled r above **0.487** against next-season actual wOBA, with the gap-CI clearly excluding 0.

But honestly, raw accuracy was never the whole point. Once I saw the parity result, the project pivoted toward two questions I find more interesting: how *confident* should we be in a given hitter's number, and what's their *true talent* underneath the noise? Those are Parts 3 and 4.